# 02 · The controller check
**Idea:** a model learns, from normal operation only, what the Siemens level controller commands to the
valve and pump given the water level, its setpoints, flow, pressure and the hydro-turbine demand over the
last 30 seconds. When an attacker changes or replays a command, the actual command stops matching the
expected one.

**Rules fixed before testing:**
1. Learn from normal files only: HAI 22.04 train1-5 and 23.05 train1-3.
2. Set the alarm level on held-out normal files: 22.04 train6 and 23.05 train4 (99.9th percentile).
3. Alarm when the 10-second median of the gap stays above that level for 10 seconds.
4. Test once on all six test files. An attack counts as caught if an alarm starts during it or up to 60 s after.
5. A false alarm is any alarm that starts during attack-free time.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

In [2]:
import json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.data_loading import load_hai, FILES, attack_episodes, ATTACK_TYPES, PROCESSED
from src.water_model import ControllerCheck, evaluate, TARGETS, RULE
fit = pd.concat([load_hai("22.04", FILES[("22.04", "train")][:5]), load_hai("23.05", FILES[("23.05", "train")][:3])], ignore_index=True)
cal = pd.concat([load_hai("22.04", ["train6"]), load_hai("23.05", ["hai-train4"])], ignore_index=True)
test = pd.concat([load_hai("22.04", FILES[("22.04", "test")]), load_hai("23.05", FILES[("23.05", "test")])], ignore_index=True)
eps = attack_episodes(test)
print(f"learn from {len(fit)/3600:.0f} h | calibrate on {len(cal)/3600:.0f} h | test {len(test)/3600:.0f} h with {len(eps)} attacks")
print(eps.target.value_counts().to_dict())

learn from 401 h | calibrate on 127 h | test 179 h with 110 attacks
{'elsewhere': 97, 'water': 11, 'ambiguous': 2}


In [3]:
t0 = time.time()
model = ControllerCheck().fit(fit).calibrate(cal)
score = model.score(test); alarm = model.alarms(test, score)
res, fa = evaluate(test, alarm, eps)
print(f"trained and tested in {time.time() - t0:.0f} s on IBM LinuxONE | alarm level {model.threshold:.2f}")
water = res[res.target == "water"].copy()
water["attack"] = water.scenario.map(lambda s: " + ".join(ATTACK_TYPES[x] for x in s.split("+")))
water[["id", "version", "seconds", "attack", "also_hits", "caught", "seconds_to_flag"]]

trained and tested in 19 s on IBM LinuxONE | alarm level 6.60


,id,version,seconds,attack,also_hits,caught,seconds_to_flag
3,A104,22.04,55,"Valve command pushed up or down, then restored",NaN,True,22.0
11,A205,22.04,56,"Valve command pushed up or down, then restored",NaN,True,19.0
18,A212,22.04,56,Pump command replaced with an earlier value,NaN,True,18.0
20,A214,22.04,614,Valve command changed for more than 10 minutes,NaN,True,17.0
26,A303,22.04,144,"Valve command pushed up or down, then restored",boiler,True,24.0
27,A304,22.04,92,Valve command changed while the level reading ...,NaN,True,20.0
33,A310,22.04,2025,Pump command replaced with an earlier value + ...,NaN,True,19.0
37,A404,22.04,98,"Valve command pushed up or down, then restored",turbine,True,19.0
47,A414,22.04,78,Pump command changed while the level reading i...,turbine,True,29.0
52,A419,22.04,164,Pump command replaced with an earlier value,boiler,True,27.0


In [4]:
elsewhere = res[res.target == "elsewhere"]; amb = res[res.target == "ambiguous"]
summary = {
    "water_caught": int(water.caught.sum()), "water_attacks": int(len(water)),
    "median_seconds_to_flag": float(water.seconds_to_flag.median()),
    "elsewhere_alarmed": int(elsewhere.caught.sum()), "elsewhere_attacks": int(len(elsewhere)),
    "ambiguous": amb[["id", "caught", "seconds_to_flag"]].to_dict("records"), **fa}
print(f"Water-process attacks caught: {summary['water_caught']} of {summary['water_attacks']}, median {summary['median_seconds_to_flag']:.0f} s to flag")
print(f"Alarms during the {summary['elsewhere_attacks']} attacks on the boiler and turbine: {summary['elsewhere_alarmed']}")
print(f"False alarms: {fa['false_alarms']} in {fa['attack_free_hours']} h of attack-free time = {fa['per_day']}/day (95% range {fa['per_day_range'][0]} to {fa['per_day_range'][1]})")
print("Ambiguous table entries (not counted):", summary["ambiguous"])
by_version = {}
for v in ["22.04", "23.05"]:
    m = test.version.to_numpy() == v
    sub = test[m].reset_index(drop=True)
    r_v, fa_v = evaluate(sub, alarm[m], attack_episodes(sub))
    w_v = r_v[r_v.target == "water"]
    by_version[v] = {"water_caught": int(w_v.caught.sum()), "water_attacks": int(len(w_v)), "per_day": fa_v["per_day"],
                     "per_day_range": fa_v["per_day_range"], "false_alarms": fa_v["false_alarms"], "attack_free_hours": fa_v["attack_free_hours"]}
pd.DataFrame(by_version).T

Water-process attacks caught: 11 of 11, median 20 s to flag
Alarms during the 97 attacks on the boiler and turbine: 0
False alarms: 9 in 172.8 h of attack-free time = 1.25/day (95% range 0.57 to 2.37)
Ambiguous table entries (not counted): [{'id': 'A301', 'caught': False, 'seconds_to_flag': nan}, {'id': 'A305', 'caught': False, 'seconds_to_flag': nan}]


,water_caught,water_attacks,per_day,per_day_range,false_alarms,attack_free_hours
22.04,10,10,1.73,"[0.7, 3.57]",7,97.0
23.05,1,1,0.63,"[0.08, 2.29]",2,75.8


## For comparison: the same model trained on HAI 22.04 only
This is the earlier result (before adding 23.05's normal data). Tested on the same six files, so the
23.05 files show how a model trained a year earlier holds up.

In [5]:
model_2204 = ControllerCheck().fit(load_hai("22.04", FILES[("22.04", "train")][:5])).calibrate(load_hai("22.04", ["train6"]))
res_b, fa_b = evaluate(test, model_2204.alarms(test), eps)
wb = res_b[res_b.target == "water"]
comparison = {"water_caught": int(wb.caught.sum()), "water_attacks": int(len(wb)), "elsewhere_alarmed": int(res_b[res_b.target == "elsewhere"].caught.sum()),
              "false_alarms": fa_b["false_alarms"], "per_day": fa_b["per_day"], "per_day_range": fa_b["per_day_range"]}
print(f"22.04-only model: caught {comparison['water_caught']} of {comparison['water_attacks']} | false alarms {comparison['per_day']}/day (95% range {comparison['per_day_range'][0]} to {comparison['per_day_range'][1]}) | boiler/turbine attacks alarmed {comparison['elsewhere_alarmed']}")

22.04-only model: caught 11 of 11 | false alarms 7.78/day (95% range 5.87 to 10.1) | boiler/turbine attacks alarmed 3


## The same idea as a neural network on the Telum accelerator
A small neural network learns the same expected commands. It runs on the IBM z17's on-chip AI
accelerator (NNPA) through PyTorch. Same features, same alarm rule.

In [6]:
import copy, torch, torch_nnpa
from src.water_model import features, segments
Ff, Fc, Ft = features(fit), features(cal), features(test)
mu, sd = Ff.mean(), Ff.std().replace(0, 1)
ym, ys = fit[TARGETS].mean(), fit[TARGETS].std().replace(0, 1)
Z = lambda F: ((F - mu) / sd).clip(-20, 20).astype("float32").to_numpy()
Xf, Xc, Xt = Z(Ff), Z(Fc), Z(Ft)
yf = ((fit[TARGETS] - ym) / ys).astype("float32").to_numpy()
torch.manual_seed(0)
net = torch.nn.Sequential(torch.nn.Linear(Xf.shape[1], 64), torch.nn.ReLU(), torch.nn.Linear(64, 32), torch.nn.ReLU(), torch.nn.Linear(32, 2))
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
xt_, yt_ = torch.from_numpy(Xf[::3]), torch.from_numpy(yf[::3])
for epoch in range(8):
    perm = torch.randperm(len(xt_))
    for i in range(0, len(xt_), 2048):
        b = perm[i:i + 2048]; loss = torch.mean((net(xt_[b]) - yt_[b]) ** 2); opt.zero_grad(); loss.backward(); opt.step()
net.eval()
def predict(X, device):
    m = copy.deepcopy(net).to(device) if device != "cpu" else net
    out = []
    with torch.no_grad():
        for i in range(0, len(X), 8192):
            xb = torch.from_numpy(X[i:i + 8192])
            out.append(m(xb.to(device)).cpu().numpy() if device != "cpu" else m(xb).numpy())
    return np.concatenate(out) * ys.to_numpy() + ym.to_numpy()
t = time.time(); p_cpu = predict(Xt, "cpu"); cpu_s = time.time() - t
t = time.time(); p_nnpa = predict(Xt, "nnpa"); nnpa_s = time.time() - t
print(f"Expected commands for {len(Xt):,} test seconds: CPU {cpu_s:.2f} s, Telum accelerator {nnpa_s:.2f} s, max difference {np.abs(p_cpu - p_nnpa).max():.3f}")
def nn_alarm(df, X):
    p = predict(X, "nnpa"); return p
pc = predict(Xc, "nnpa")
sdr = {t_: float(np.std(cal[t_].to_numpy() - pc[:, k])) for k, t_ in enumerate(TARGETS)}
def nn_score(df, p):
    raw = np.max([np.abs(df[t_].to_numpy() - p[:, k]) / sdr[t_] for k, t_ in enumerate(TARGETS)], axis=0)
    return pd.Series(raw).groupby(segments(df)).transform(lambda x: x.rolling(RULE["smooth_s"], min_periods=1).median()).to_numpy()
nn_model = ControllerCheck(); nn_model.threshold = float(np.quantile(nn_score(cal, pc), RULE["quantile"]))
res_n, fa_n = evaluate(test, nn_model.alarms(test, nn_score(test, p_nnpa)), eps)
wn = res_n[res_n.target == "water"]
accelerator = {"rows": int(len(Xt)), "cpu_seconds": round(cpu_s, 2), "nnpa_seconds": round(nnpa_s, 2),
               "water_caught": int(wn.caught.sum()), "water_attacks": int(len(wn)), "per_day": fa_n["per_day"], "per_day_range": fa_n["per_day_range"],
               "elsewhere_alarmed": int(res_n[res_n.target == "elsewhere"].caught.sum())}
print(f"Neural network on Telum: caught {accelerator['water_caught']} of {accelerator['water_attacks']} | false alarms {accelerator['per_day']}/day (95% range {accelerator['per_day_range'][0]} to {accelerator['per_day_range'][1]}) | boiler/turbine attacks alarmed {accelerator['elsewhere_alarmed']}")

Expected commands for 645,600 test seconds: CPU 0.22 s, Telum accelerator 0.08 s, max difference 56.194


Neural network on Telum: caught 10 of 11 | false alarms 0.56/day (95% range 0.15 to 1.42) | boiler/turbine attacks alarmed 0


## Save results for the dashboard

In [7]:
exp = model.expected(test)
traces = []
for _, e in water.iterrows():
    a, b = max(0, e.i0 - 120), min(len(test), e.i1 + 181)
    step = max(1, (b - a) // 600)
    idx = np.arange(a, b, step)
    sec = (test.timestamp.to_numpy()[idx] - test.timestamp.to_numpy()[e.i0]) / np.timedelta64(1, "s")
    traces.append({"id": e.id, "version": e.version, "start": str(e.start)[:19], "seconds": int(e.seconds), "scenario": e.scenario,
                   "attack_text": e.attack, "also_hits": e.also_hits, "caught": bool(e.caught), "seconds_to_flag": e.seconds_to_flag,
                   "t": sec.round(0).tolist(), "attack_flag": test.attack.to_numpy()[idx].tolist(),
                   "level": test.P3_LIT01.to_numpy()[idx].round(2).tolist(),
                   "valve": test.P3_LCV01D.to_numpy()[idx].round(1).tolist(), "valve_expected": exp.P3_LCV01D.to_numpy()[idx].round(1).tolist(),
                   "pump": test.P3_LCP01D.to_numpy()[idx].round(1).tolist(), "pump_expected": exp.P3_LCP01D.to_numpy()[idx].round(1).tolist(),
                   "score": (score[idx] / model.threshold).round(3).tolist()})
PROCESSED.mkdir(parents=True, exist_ok=True)
results = {"rule": RULE, "threshold": model.threshold,
           "hours": {"learn": round(len(fit) / 3600, 1), "calibrate": round(len(cal) / 3600, 1), "test": round(len(test) / 3600, 1)},
           "summary": summary, "by_version": by_version, "comparison_2204_only": comparison, "neural_network_on_telum": accelerator,
           "water_attacks": water[["id", "version", "start", "seconds", "scenario", "attack", "also_hits", "caught", "seconds_to_flag"]].astype({"start": str}).to_dict("records"),
           "elsewhere_counts": {v: int(((res.target == "elsewhere") & (res.version == v)).sum()) for v in ["22.04", "23.05"]}}
(PROCESSED / "controller_check_results.json").write_text(json.dumps(results, default=str))
(PROCESSED / "controller_check_traces.json").write_text(json.dumps(traces, default=str))
print("saved results and", len(traces), "attack traces")

saved results and 11 attack traces
